# Factoring 15

In [ ]:
import numpy as np
from math import gcd
from fractions import Fraction

from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.circuit.library import QFTGate, UnitaryGate
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator

import matplotlib.pyplot as plt

# Setting

In [ ]:
N = 15
a = 2

L = 4 # The target register needs 4 qubits because 15 < 2^4
n_count = 8 # Number of counting qubits for phase precision
Q = 2**n_count

In [ ]:
def mul_mod_N_gate(c: int, N: int = 15, L: int = 4) -> UnitaryGate:
    """
    Build a unitary gate implementing |y> -> |c*y mod N>.
    For demonstration, states outside 0 <= y < N are left unchanged.
    c = a^{2^{j}} mod N
    """
    dim = 2**L
    U = np.zeros((dim, dim), dtype=complex)

    for y in range(dim):
        if y < N:
            out = (c * y) % N
        else:
            out = y
        U[out, y] = 1.0

    return UnitaryGate(U, label=f"*{c} mod {N}")

In [ ]:
plt.matshow(mul_mod_N_gate(c=2, N=15).params[0].real)

## Construct the quantum order-finding circuit used in Shor's algorithm

In [ ]:
counting = list(range(n_count))
target = list(range(n_count, n_count + L))

In [ ]:
qc = QuantumCircuit(n_count + L, n_count)

# Initialize the target register to |1>
qc.x(target[0])
qc.barrier()

# Put the counting register into a uniform superposition
qc.h(counting)
qc.barrier()

# Apply controlled-U^(2^j), where U|y> = |a*y mod N>
for j in range(n_count):
    c = pow(base=a, exp=(2**j), mod=N)
    u = mul_mod_N_gate(c, N=N, L=L).control(1, annotated=False)
    qc.append(u, [counting[j]] + target)
qc.barrier()

# Apply the inverse QFT to extract the phase information
qc.append(QFTGate(n_count).inverse(annotated=True), counting)
qc.barrier()

# Measure the counting register
qc.measure(counting, counting)

In [ ]:
qc.draw(output='mpl')

In [ ]:
simulator = AerSimulator()

In [ ]:
pass_manager = generate_preset_pass_manager(backend=simulator, optimization_level=1)

In [ ]:
transpiled_qc = pass_manager.run(qc)

In [ ]:
# transpiled_qc.draw(output='text')

In [ ]:
job = simulator.run(transpiled_qc, shots=2048, seed_simulator=1234)

In [ ]:
result = job.result()

In [ ]:
counts = result.get_counts()

In [ ]:
plot_histogram(counts)

# Postprocess
Use continued fractions and gcd computations to extract the factors

In [ ]:
print("Measured candidates:")
for bitstring, shots in sorted(counts.items(), key=lambda kv: kv[1], reverse=True):
    m = int(bitstring, base=2)
    phase = Fraction(m, Q).limit_denominator(N)
    r = phase.denominator

    print(
        f"  bitstring={bitstring}, shots={shots:4d}, "
        f"m={m:3d}, phase≈{phase}, candidate r={r}"
    )

    # Skip the trivial phase
    if phase.numerator == 0:
        continue

    # The order must be even for the standard factor extraction step
    if r % 2 != 0:
        continue

    # Check that r is a valid order candidate
    if pow(a, r, N) != 1:
        continue

    x = pow(a, r // 2, N)

    # If x = -1 mod N, this attempt does not give non-trivial factors
    if x == N - 1:
        continue

    # Extract non-trivial factors using gcd
    p = gcd(x - 1, N)
    q = gcd(x + 1, N)

    if 1 < p < N and 1 < q < N:
        print(f"🙌 phase={phase} --> {r=}")
        print(f"🙌 {N=} --> {p=} & {q=}")
        break
else:
    print("😱 No non-trivial factor found. Try more shots or rerun.")